Bài 1

In [ ]:
import sqlite3
import math  

# Kết nối database và bật tính năng hàm toán học
conn = sqlite3.connect('data.db')
conn.create_function('SQRT', 1, math.sqrt)  
cursor = conn.cursor()

# Tạo bảng và chèn dữ liệu mẫu
cursor.execute('''CREATE TABLE IF NOT EXISTS data_table 
                 (id INTEGER PRIMARY KEY, A REAL, B REAL)''')

sample_data = [
    (1, 10, 20),
    (2, 15, 25),
    (3, 20, 30),
    (4, 25, 35),
    (5, 30, 40)
]
cursor.executemany('INSERT INTO data_table (id, A, B) VALUES (?, ?, ?)', sample_data)
conn.commit()

# Tính toán trong Python sau khi lấy dữ liệu từ SQL
query_basic = '''
SELECT 
    SUM(A) as sum_a,
    SUM(B) as sum_b,
    SUM(A*B) as sum_ab,
    SUM(A*A) as sum_a2,
    SUM(B*B) as sum_b2,
    COUNT(*) as n
FROM data_table
'''

cursor.execute(query_basic)
data = cursor.fetchone()

# Tính toán hệ số tương quan
numerator = data[5] * data[2] - data[0] * data[1]
denominator = math.sqrt(data[5] * data[3] - data[0]**2) * math.sqrt(data[5] * data[4] - data[1]**2)
correlation = numerator / denominator

print(f"Hệ số tương quan giữa A và B là: {correlation:.4f}")

# Đóng kết nối
conn.close()

Hệ số tương quan giữa A và B là: 1.0000


Bài 2

In [4]:
import sqlite3
import pandas as pd
from scipy.stats import chi2_contingency

# 1. TẠO DATABASE VÀ CHÈN DỮ LIỆU
conn = sqlite3.connect('car.db')
cursor = conn.cursor()

# Tạo bảng Scores nếu chưa có
cursor.execute('''
    CREATE TABLE IF NOT EXISTS Scores (
        Day TEXT,
        A REAL,
        B REAL,
        C REAL
    )
''')

# Dữ liệu từ hình ảnh
data = [
    ("Day 1", 8, 9, 7),
    ("Day 2", 7.5, 8.5, 7),
    ("Day 3", 6, 7, 8),
    ("Day 4", 7, 6, 5)
]

# Xoá dữ liệu cũ nếu có và chèn lại 
cursor.execute('DELETE FROM Scores')
cursor.executemany('INSERT INTO Scores (Day, A, B, C) VALUES (?, ?, ?, ?)', data)
conn.commit()

# 2. ĐỌC VÀ CHUYỂN DỮ LIỆU SANG DẠNG QUAN HỆ
df = pd.read_sql_query("SELECT * FROM Scores", conn)

# Chuyển sang long format: Day | Person | Score
df_melted = df.melt(id_vars=["Day"], var_name="Person", value_name="Score")

# Phân loại điểm
def categorize(score):
    if score >= 8:
        return "Cao"
    elif score >= 6:
        return "Trung bình"
    else:
        return "Thấp"

df_melted["ScoreLevel"] = df_melted["Score"].apply(categorize)

# 3. KIỂM ĐỊNH CHI-SQUARE: PERSON vs SCORE LEVEL
contingency_person = pd.crosstab(df_melted["Person"], df_melted["ScoreLevel"])
chi2_person, p_person, dof_person, expected_person = chi2_contingency(contingency_person)

print("📊 Bảng tần suất (Person vs ScoreLevel):")
print(contingency_person)
print("\n📈 Kết quả kiểm định χ² cho Mẫu xe:")
print(f"Chi2 statistic = {chi2_person:.4f}")
print(f"Degrees of freedom = {dof_person}")
print(f"P-value = {p_person:.4f}")

# 4. KIỂM ĐỊNH CHI-SQUARE: DAY vs SCORE LEVEL
contingency_day = pd.crosstab(df_melted["Day"], df_melted["ScoreLevel"])
chi2_day, p_day, dof_day, expected_day = chi2_contingency(contingency_day)

print("\n📊 Bảng tần suất (Day vs ScoreLevel):")
print(contingency_day)
print("\n📈 Kết quả kiểm định χ² cho Ngày:")
print(f"Chi2 statistic = {chi2_day:.4f}")
print(f"Degrees of freedom = {dof_day}")
print(f"P-value = {p_day:.4f}")

# 5. KẾT LUẬN
print("\n📌 KẾT LUẬN:")

if p_person < 0.05:
    print(" Kết quả thử nghiệm PHỤ THUỘC vào mẫu xe (Person).")
else:
    print(" Kết quả thử nghiệm KHÔNG phụ thuộc vào mẫu xe (Person).")

if p_day < 0.05:
    print(" Kết quả thử nghiệm PHỤ THUỘC vào ngày (Day).")
else:
    print(" Kết quả thử nghiệm KHÔNG phụ thuộc vào ngày (Day).")

# Đóng kết nối
conn.close()


📊 Bảng tần suất (Person vs ScoreLevel):
ScoreLevel  Cao  Thấp  Trung bình
Person                           
A             1     0           3
B             2     0           2
C             1     1           2

📈 Kết quả kiểm định χ² cho Mẫu xe:
Chi2 statistic = 2.7857
Degrees of freedom = 4
P-value = 0.5943

📊 Bảng tần suất (Day vs ScoreLevel):
ScoreLevel  Cao  Thấp  Trung bình
Day                              
Day 1         2     0           1
Day 2         1     0           2
Day 3         1     0           2
Day 4         0     1           2

📈 Kết quả kiểm định χ² cho Ngày:
Chi2 statistic = 5.4286
Degrees of freedom = 6
P-value = 0.4901

📌 KẾT LUẬN:
 Kết quả thử nghiệm KHÔNG phụ thuộc vào mẫu xe (Person).
 Kết quả thử nghiệm KHÔNG phụ thuộc vào ngày (Day).


bài 3

In [11]:
import sqlite3

# 1. Kết nối CSDL và tạo bảng mẫu
conn = sqlite3.connect('flights.db')  
cursor = conn.cursor()

# Tạo bảng flights với dữ liệu mẫu
cursor.execute("""
CREATE TABLE flights (
    id INTEGER PRIMARY KEY,
    departure_time INTEGER
)
""")

# Thêm dữ liệu mẫu (830, 1445, 30, 1200, 2359)
sample_times = [(830,), (1445,), (30,), (1200,), (2359,)]
cursor.executemany("INSERT INTO flights (departure_time) VALUES (?)", sample_times)
conn.commit()

# 2. Hàm chuyển đổi đơn giản trong Python
def format_time(time_int):
    """Chuyển số nguyên thành chuỗi thời gian dạng HH:MM AM/PM"""
    time_str = f"{time_int:04d}"  # Đảm bảo có 4 chữ số (30 -> 0030)
    hour = int(time_str[:2])
    minute = time_str[2:]
    
    # Xác định AM/PM và chuyển sang giờ 12
    period = 'AM' if hour < 12 else 'PM'
    hour_12 = hour % 12
    hour_12 = 12 if hour_12 == 0 else hour_12  # 0 giờ thành 12 AM
    
    return f"{hour_12}:{minute} {period}"

# 3. Truy vấn và chuyển đổi
cursor.execute("SELECT departure_time FROM flights")
results = cursor.fetchall()

print("Kết quả chuyển đổi:")
print("{:<10} {:<10}".format("Số nguyên", "Thời gian"))
for row in results:
    original = row[0]
    formatted = format_time(original)
    print("{:<10} {:<10}".format(original, formatted))

# 4. Đóng kết nối
conn.close()

Kết quả chuyển đổi:
Số nguyên  Thời gian 
830        8:30 AM   
1445       2:45 PM   
30         12:30 AM  
1200       12:00 PM  
2359       11:59 PM  


bài 4

In [1]:
import sqlite3
import statistics

def tim_gia_tri_ngoai_le():
    # 1. Kết nối database và tạo bảng nếu chưa có
    conn = sqlite3.connect('du_lieu.db')
    cursor = conn.cursor()
    
    # Tạo bảng và thêm dữ liệu mẫu (bỏ qua nếu đã có dữ liệu thật)
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS du_lieu (
        gia_tri REAL
    )
    """)
    
    # Thêm vài dữ liệu mẫu (có 1 giá trị ngoại lệ là 100)
    cursor.executemany("INSERT INTO du_lieu VALUES (?)", 
                      [(12.5,), (13.2,), (12.8,), (100.0,), (11.9,)])
    conn.commit()
    
    # 2. Lấy tất cả giá trị từ database
    cursor.execute("SELECT gia_tri FROM du_lieu")
    cac_gia_tri = [row[0] for row in cursor.fetchall()]
    
    # 3. Tính toán các giá trị thống kê
    trung_vi = statistics.median(cac_gia_tri)
    
    # Tính MAD (độ lệch tuyệt đối trung vị)
    do_lech = [abs(x - trung_vi) for x in cac_gia_tri]
    mad = statistics.median(do_lech)
    
    # Ngưỡng ngoại lệ (1.5 lần MAD)
    nguong = 1.5 * mad
    
    # 4. Tìm các giá trị ngoại lệ
    ngoai_le = [x for x in cac_gia_tri if abs(x - trung_vi) > nguong]
    
    # 5. Hiển thị kết quả
    print("\nKẾT QUẢ PHÂN TÍCH")
    print(f"- Số lượng mẫu: {len(cac_gia_tri)}")
    print(f"- Giá trị trung vị: {trung_vi:.2f}")
    print(f"- Độ lệch MAD: {mad:.2f}")
    print(f"- Ngưỡng ngoại lệ: ±{nguong:.2f}")
    
    if ngoai_le:
        print("\nCÁC GIÁ TRỊ NGOẠI LỆ:")
        for gia_tri in ngoai_le:
            print(f"- {gia_tri:.2f} (lệch {abs(gia_tri - trung_vi):.2f} so với trung vị)")
    else:
        print("\nKhông tìm thấy giá trị ngoại lệ")
    
    conn.close()

# Chạy chương trình
tim_gia_tri_ngoai_le()


KẾT QUẢ PHÂN TÍCH
- Số lượng mẫu: 5
- Giá trị trung vị: 12.80
- Độ lệch MAD: 0.40
- Ngưỡng ngoại lệ: ±0.60

CÁC GIÁ TRỊ NGOẠI LỆ:
- 100.00 (lệch 87.20 so với trung vị)
- 11.90 (lệch 0.90 so với trung vị)


Bài 5

In [ ]:
import sqlite3

def check_duplicate_patients():
    # Kết nối database và tạo bảng
    conn = sqlite3.connect("patients.db")
    cursor = conn.cursor()

    # Tạo bảng Patient 
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS Patient (
        id INTEGER PRIMARY KEY,
        last_name TEXT,
        weight REAL,
        height REAL
    )
    """)

    # Thêm dữ liệu mẫu 
    sample_data = [
        (1, "Nguyen", 65.0, 1.70),
        (2, "Tran", 70.5, 1.75),
        (3, "Nguyen", 65.5, 1.68),  
        (4, "Le", 68.0, 1.72),
        (5, "Nguyen", 65.2, 1.71)   
    ]
    cursor.executemany("INSERT INTO Patient VALUES (?, ?, ?, ?)", sample_data)
    conn.commit()

    # Truy vấn tìm các cặp có thể là một người
    query = """
    SELECT 
        a.id AS id1, 
        b.id AS id2, 
        a.last_name, 
        a.weight AS weight1, 
        b.weight AS weight2
    FROM 
        Patient a, Patient b
    WHERE 
        a.id < b.id  -- Tránh trùng lặp và so sánh ngược
        AND a.last_name = b.last_name  -- Điều kiện Boolean cho họ
        AND ABS(a.weight - b.weight) <= 1.0  -- Điều kiện Boolean cho cân nặng
    """

    cursor.execute(query)
    duplicates = cursor.fetchall()

    # Hiển thị kết quả
    print("CÁC CẶP BỆNH NHÂN CÓ THỂ LÀ MỘT NGƯỜI:")
    if duplicates:
        for dup in duplicates:
            print(f"- ID {dup[0]} và ID {dup[1]}:")
            print(f"  + Họ: {dup[2]}")
            print(f"  + Cân nặng: {dup[3]}kg vs {dup[4]}kg (chênh lệch: {abs(dup[3] - dup[4])}kg)")
    else:
        print("Không tìm thấy cặp trùng lặp nào.")

    conn.close()

# Chạy chương trình
check_duplicate_patients()

CÁC CẶP BỆNH NHÂN CÓ THỂ LÀ MỘT NGƯỜI:
- ID 1 và ID 5:
  + Họ: Nguyen
  + Cân nặng: 65.0kg vs 65.2kg (chênh lệch: 0.20000000000000284kg)
- ID 1 và ID 3:
  + Họ: Nguyen
  + Cân nặng: 65.0kg vs 65.5kg (chênh lệch: 0.5kg)
- ID 3 và ID 5:
  + Họ: Nguyen
  + Cân nặng: 65.5kg vs 65.2kg (chênh lệch: 0.29999999999999716kg)
